# LLM-from-Scratch (124M Parameters)

Train a GPT-style Transformer from scratch on Google Colab using FineWeb-Edu dataset.

**Features:**
- **~124M parameters** decoder-only Transformer
- **tiktoken (GPT-2)** tokenizer — battle-tested BPE
- **FineWeb-Edu** dataset — high-quality web text
- **Perplexity tracking** + sample generation during training
- **Session-safe** with automatic checkpoint resume
- **torch.compile** for ~1.5× speedup on T4/A100

> **Tip:** Enable GPU (T4/A100) before starting: Runtime → Change runtime type → GPU

## 1. Verify GPU & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q torch numpy tqdm requests matplotlib regex datasets tiktoken huggingface-hub

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Mount Google Drive & Clone Repository

> Drive is used to persist checkpoints and logs across Colab sessions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/llm-from-scratch/checkpoints
!mkdir -p /content/drive/MyDrive/llm-from-scratch/logs

%cd /content
!rm -rf llm-from-scratch
!git clone https://github.com/avneeshjadhav04/llm-from-scratch.git
%cd llm-from-scratch

## 3. Prepare Data (FineWeb-Edu)

> Configure the number of tokens and train/val split below.
> Default: 200M tokens (95/5 train/val split).
> **Takes ~20–40 minutes on first run.** Subsequent runs skip if binaries exist.

In [ ]:
# --- Data Preparation Configuration ---
NUM_TOKENS = 200_000_000     # Total tokens to download & tokenize
TRAIN_SPLIT = 0.95           # Train/validation split ratio (0.0–1.0)

print(f"Data prep config:")
print(f"  Target tokens:   {NUM_TOKENS:,}")
print(f"  Train split:     {TRAIN_SPLIT}")

!python prepare_data.py \
    --num_tokens {NUM_TOKENS} \
    --train_split {TRAIN_SPLIT}

## 4. Resume Setup — Copy from Drive

> Checks Drive for existing checkpoints and logs, then copies them locally.
> Run this before every training session (fresh or resume).

In [ ]:
import os
import glob

DRIVE_CKPT_DIR = "/content/drive/MyDrive/llm-from-scratch/checkpoints"
DRIVE_LOG_DIR = "/content/drive/MyDrive/llm-from-scratch/logs"

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("logs", exist_ok=True)

drive_checkpoints = sorted(glob.glob(f"{DRIVE_CKPT_DIR}/*.pt"))
if drive_checkpoints:
    latest = drive_checkpoints[-1]
    step = int(os.path.basename(latest).split("_")[-1].replace(".pt", ""))
    print(f"Found checkpoint in Drive: {os.path.basename(latest)} (step {step})")
    print("Copying checkpoints and logs from Drive...")
    !cp {DRIVE_CKPT_DIR}/*.pt checkpoints/ 2>/dev/null || true
    !cp {DRIVE_LOG_DIR}/* logs/ 2>/dev/null || true
    print(f"Ready to resume from step {step}")
else:
    print("No checkpoint found in Drive. Training will start from scratch.")

## 5. Train the 124M Model

> Adjust the hyperparameters below, then run the cell.
> Training will resume automatically if a checkpoint exists locally (copied in Step 4).

In [ ]:
# --- Training Configuration ---
# Adjust these values before each training session

BATCH_SIZE = 16              # Reduce if OOM (e.g., 8 or 4)
LEARNING_RATE = 6e-4         # Default: 6e-4
SESSION_STEPS = 5000         # Steps to run THIS session (not total)
MAX_SEQ_LEN = 512            # Context length (no need to re-prepare data if changed)
WARMUP_STEPS = 2000          # LR warmup steps
USE_COMPILE = True           # torch.compile for speed (set False if unstable)

print(f"Training config:")
print(f"  Batch size:      {BATCH_SIZE}")
print(f"  Learning rate:   {LEARNING_RATE}")
print(f"  Session steps:   {SESSION_STEPS}")
print(f"  Max seq length:  {MAX_SEQ_LEN}")
print(f"  Warmup steps:    {WARMUP_STEPS}")
print(f"  torch.compile:   {USE_COMPILE}")

!python train.py \
    --batch_size {BATCH_SIZE} \
    --learning_rate {LEARNING_RATE} \
    --max_steps_per_session {SESSION_STEPS} \
    --max_seq_len {MAX_SEQ_LEN} \
    --warmup_steps {WARMUP_STEPS} \
    --compile {USE_COMPILE}

## 6. Save to Drive

> **Run this even if training was interrupted!**
> By default it auto-detects the latest checkpoint. Override below if needed.

In [ ]:
import os
import glob
import shutil

# --- Save Configuration ---
# Leave as 0 to auto-detect the latest checkpoint, or enter a specific step number
SAVE_UP_TO_STEP = 0   # e.g., 5000

DRIVE_CKPT_DIR = "/content/drive/MyDrive/llm-from-scratch/checkpoints"
DRIVE_LOG_DIR = "/content/drive/MyDrive/llm-from-scratch/logs"

# Find checkpoints
local_ckpts = sorted(glob.glob("checkpoints/*.pt"))
if not local_ckpts:
    print("No local checkpoints found. Nothing to save.")
else:
    if SAVE_UP_TO_STEP > 0:
        target_ckpt = None
        for ckpt in local_ckpts:
            step = int(os.path.basename(ckpt).split("_")[-1].replace(".pt", ""))
            if step <= SAVE_UP_TO_STEP:
                target_ckpt = ckpt
        if target_ckpt is None:
            print(f"WARNING: No checkpoint found at or before step {SAVE_UP_TO_STEP}")
            print("Falling back to latest checkpoint...")
            target_ckpt = local_ckpts[-1]
        else:
            print(f"Saving checkpoint up to step {SAVE_UP_TO_STEP}")
    else:
        target_ckpt = local_ckpts[-1]
        step = int(os.path.basename(target_ckpt).split("_")[-1].replace(".pt", ""))
        print(f"Auto-detected latest checkpoint: step {step}")

    # Copy selected checkpoints (skip if already on Drive with same size)
    saved_any = False
    for ckpt in local_ckpts:
        step = int(os.path.basename(ckpt).split("_")[-1].replace(".pt", ""))
        if SAVE_UP_TO_STEP > 0 and step > SAVE_UP_TO_STEP:
            continue
        dest = os.path.join(DRIVE_CKPT_DIR, os.path.basename(ckpt))
        if not os.path.exists(dest) or os.path.getsize(ckpt) != os.path.getsize(dest):
            shutil.copy(ckpt, DRIVE_CKPT_DIR)
            print(f"  Saved checkpoint: {os.path.basename(ckpt)}")
            saved_any = True
    if not saved_any:
        print("  All checkpoints already up to date on Drive.")

    # Save logs
    log_files = glob.glob("logs/*")
    if log_files:
        for f in log_files:
            shutil.copy(f, DRIVE_LOG_DIR)
            print(f"  Saved log: {os.path.basename(f)}")
    else:
        print("No local logs found.")

    print("\nDone! All outputs saved to Drive.")

## 7. Generate Text (Interactive)

> Uses the **latest saved checkpoint** automatically.

In [ ]:
!python generate.py \
    --prompt "The future of artificial intelligence is" \
    --temperature 0.8 \
    --top_k 40 \
    --top_p 0.95 \
    --max_new_tokens 256 \
    --max_seq_len {MAX_SEQ_LEN} \
    --device cuda

## 8. Plot Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("logs/100m_training_log.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df["step"], df["train_loss"], label="Train Loss")
val_mask = df["val_loss"].notna()
axes[0].plot(df.loc[val_mask, "step"], df.loc[val_mask, "val_loss"], label="Val Loss", linestyle="--")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(df["step"], df["train_ppl"], label="Train PPL")
axes[1].plot(df.loc[val_mask, "step"], df.loc[val_mask, "val_ppl"], label="Val PPL", linestyle="--")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Perplexity")
axes[1].set_title("Training & Validation Perplexity")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()